Structured output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures

In [40]:
import os
from langchain.chat_models import init_chat_model
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:openai/gpt-oss-120b")
model


ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F61D875B50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F61D876150>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [41]:


from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(description="the title of the movie")
    year:int=Field(description="the movie released in this year")
    director:str=Field(description="director of the movie")
    rating:float=Field(description="the movie imdb rating")


In [42]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x000001F61D875B50>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001F61D876150>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'the t

In [43]:
model.invoke("provide me the details of the movie 96")

AIMessage(content='**“96” – 2018 Indian Tamil‑language Romantic Drama**\n\n| Item | Details |\n|------|---------|\n| **Title** | 96 (pronounced “Ninety‑Six”) |\n| **Release Date** | 23\u202fJune\u202f2018 (India) |\n| **Language** | Tamil (dubbed in Telugu as *96*, and later released with subtitles in several other languages) |\n| **Running Time** | 2\u202fh\u202f30\u202fmin (150\u202fminutes) |\n| **Genre** | Romance, Drama, Musical |\n| **Director / Writer** | **C.\u202fVijayakumar** (debut as a solo director; previously co‑wrote *Mundasupatti* and *Jigarthanda*) |\n| **Producer** | **R.\u202fB.\u202fChoudary** –\u202fSukumar\u202fGroup (under the banner *AVM Productions* for the Tamil version) |\n| **Production Companies** | AVM Productions (Tamil) – co‑production with **Sukumar Studios** (Telugu) |\n| **Cinematography** | **R.\u202fMadhusudhan** |\n| **Editing** | **R.\u202fM.\u202fVijay** |\n| **Music Composer** | **Govind Vasantha** (also known as Govind Vasantha of the band *Tha

In [44]:
model_with_structure.invoke("provide me the details of the movie 96")

Movie(title='96', year=2018, director='C. Prem Kumar', rating=8.5)

message output alongside parse structure


In [57]:
from requests.models import Response
from langchain.chat_models import init_chat_model
import os 
os.environ['GROQ_API_KEY']=os.getenv("GROQ_API_KEY")
model=init_chat_model(model="groq:openai/gpt-oss-120b")

from pydantic import BaseModel,Field
class Movie(BaseModel):
    title:str=Field(...,description="the title of the movie")
    year:int=Field(...,description="the movie released in this year")
    director:str=Field(...,description="director of the movie")
    rating:float=Field(...,description="the movie imdb rating")
model_with_structure=model.with_structured_output(Movie,include_raw=True)
response=model_with_structure.invoke("provide details about movie salaar...")
response


{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "provide details about movie salaar...". They likely want details like director, rating, title, year. We have a function Movie that expects director, rating, title, year. We can call it with details about Salaar. Need to know actual data. Salaar is an Indian Telugu-language action thriller starring Prabhas, directed by Prashanth Neel, released in 2024? Actually Salaar release date is 2024 (maybe 2024). IMDb rating? Might be not yet released; but we can approximate. Could use placeholder? Better to fetch via function? The function just returns any; we need to call it with details. Provide details in answer after calling function. Let\'s call function with known data: director: Prashanth Neel, rating: maybe 7.5 (just guess), title: Salaar, year: 2024.', 'tool_calls': [{'id': 'fc_ea333efc-67b5-4cfe-95d2-008f0f92f69e', 'function': {'arguments': '{"director":"Prashanth Neel","rating":7.5,"title":"Salaar","year

nested structure

In [60]:
class Actor(BaseModel):
    name:str
    role:str
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genre:list[str]
    budeget:float | None = Field(None,description="budget in million usd")
model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("provide details about movie salaar")
response

MovieDetails(title='Salaar', year=2024, cast=[Actor(name='Prabhas', role='Salaar'), Actor(name='Shruti Haasan', role='Lead female'), Actor(name='Prakash Raj', role='Antagonist'), Actor(name='Jagapathi Babu', role='Supporting')], genre=['Action', 'Thriller'], budeget=None)

TypedDict

TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation

In [63]:
from typing_extensions import Annotated,TypedDict
class MovieDict(TypedDict):
    """a movie with details"""
    title:Annotated[str,...,"title of the movie"]
    year:Annotated[int,...,"year of movie release"]
    director:Annotated[str,...,"name of the movie director"]
    rating:Annotated[float,...,"rating of movie"]
model_with_typedict=model.with_structured_output(MovieDict)
response=model_with_typedict.invoke("provide details of movie salaar")
response

{'director': 'Sukumar', 'rating': 0, 'title': 'Salaar', 'year': 2024}

In [64]:
class Actor(TypedDict):
    name:str
    role:str
class MovieDetails(TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genre:list[str]
    budeget:float | None = Field(None,description="budget in million usd")
model_with_structure=model.with_structured_output(MovieDetails)
response=model_with_structure.invoke("provide details about movie salaar")
response

{'budeget': 2000000000,
 'cast': [{'name': 'Prabhas', 'role': 'Salaar'},
  {'name': 'Jagapathi Babu', 'role': 'Antagonist'},
  {'name': 'Shruti Haasan', 'role': 'Lead Actress'},
  {'name': 'Prakash Raj', 'role': 'Supporting'},
  {'name': 'Sanjay Dutt', 'role': 'Special Appearance'}],
 'genre': ['Action', 'Thriller'],
 'title': 'Salaar',
 'year': 2023}

In [65]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

Data classes

In [66]:
from pydantic import BaseModel,Field
from langchain.agents import create_agent
class contactinfo(BaseModel):
    """contact inoformation of the person"""
    name:str=Field(description="name of the person")
    email:str=Field(description="email of the person")
    phno:int=Field(description="mobile no of the person")
agent=create_agent(model="groq:openai/gpt-oss-120b",response_format=contactinfo)
result=agent.invoke({"messages":[{"role":"user","content":"extract contact from harsha,nadendlaharsha24@gmail.com,8008708670"}]})
result

{'messages': [HumanMessage(content='extract contact from harsha,nadendlaharsha24@gmail.com,8008708670', additional_kwargs={}, response_metadata={}, id='60d23451-6c64-4b27-bb0f-a8969cca9444'),
  AIMessage(content='{"name":"Harsha","email":"nadendlaharsha24@gmail.com","phno":8008708670}', additional_kwargs={'reasoning_content': 'We need to output JSON matching the schema "contactinfo". Fields: name, email, phno. All required. Provide compact JSON.\n\nUser: "extract contact from harsha,nadendlaharsha24@gmail.com,8008708670". So name: "harsha"? Probably "Harsha". email: "nadendlaharsha24@gmail.com". phno: integer 8008708670.\n\nNeed to ensure integer fits within typical range; just output number.\n\nReturn JSON: {"name":"Harsha","email":"nadendlaharsha24@gmail.com","phno":8008708670}\n\nMake sure compact, no spaces? Could be with spaces but still valid. Use compact: {"name":"Harsha","email":"nadendlaharsha24@gmail.com","phno":8008708670}\n\nReturn only JSON.'}, response_metadata={'token_us

In [67]:
result["structured_response"]

contactinfo(name='Harsha', email='nadendlaharsha24@gmail.com', phno=8008708670)

In [68]:
from typing_extensions import TypedDict
from langchain.agents import create_agent
class contactinfo(TypedDict):
    """contact inoformation of the person"""
    name:str
    email:str
    phno:int
agent=create_agent(model="groq:openai/gpt-oss-120b",response_format=contactinfo)
result=agent.invoke({"messages":[{"role":"user","content":"extract contact from harsha,nadendlaharsha24@gmail.com,8008708670"}]})
result

{'messages': [HumanMessage(content='extract contact from harsha,nadendlaharsha24@gmail.com,8008708670', additional_kwargs={}, response_metadata={}, id='f3499987-5eda-4707-97ee-333698702659'),
  AIMessage(content='{"name":"harsha","email":"nadendlaharsha24@gmail.com","phno":8008708670}', additional_kwargs={'reasoning_content': 'We need to output JSON according to schema "contactinfo". Required fields: name, email, phno. Provide compact JSON. Name: "harsha" maybe capital? Input: "harsha". Email: "nadendlaharsha24@gmail.com". Phone number: 8008708670 (integer). Ensure integer fits. Output JSON object with those fields. Use compact formatting, no extra spaces.'}, response_metadata={'token_usage': {'completion_tokens': 120, 'prompt_tokens': 214, 'total_tokens': 334, 'completion_time': 0.247697207, 'completion_tokens_details': {'reasoning_tokens': 83}, 'prompt_time': 0.009805845, 'prompt_tokens_details': None, 'queue_time': 0.377309812, 'total_time': 0.257503052}, 'model_name': 'openai/gpt-o

Dataclass

In [ ]:
from dataclasses import dataclass
@dataclass
class contactinfo:
    """contact inoformation of the person"""
    name:str
    email:str
    phno:int
agent=create_agent(model="groq:openai/gpt-oss-120b",response_format=contactinfo)
result=agent.invoke({"messages":[{"role":"user","content":"extract contact from harsha,nadendlaharsha24@gmail.com,8008708670"}]})
result
result["structured_response"]  

contactinfo(name='harsha', email='nadendlaharsha24@gmail.com', phno=8008708670)